# Datathon 2026 R1 - MCQ Notebook

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Tim project root de notebook chay duoc ca khi cwd la thu muc notebooks.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"

# Dung duong dan day du voi pd.read_csv de doc CSV on dinh.
orders = pd.read_csv(
    DATA_RAW / "transaction" / "orders.csv",
    low_memory=False,
    parse_dates=["order_date"]
)
order_items = pd.read_csv(DATA_RAW / "transaction" / "order_items.csv", low_memory=False)
payments = pd.read_csv(DATA_RAW / "transaction" / "payments.csv", low_memory=False)
returns = pd.read_csv(DATA_RAW / "transaction" / "returns.csv", low_memory=False)
products = pd.read_csv(DATA_RAW / "master" / "products.csv", low_memory=False)
customers = pd.read_csv(DATA_RAW / "master" / "customers.csv", low_memory=False)
geography = pd.read_csv(DATA_RAW / "master" / "geography.csv", low_memory=False)
web_traffic = pd.read_csv(DATA_RAW / "operational" / "web_traffic.csv", low_memory=False)
sales = pd.read_csv(
    DATA_RAW / "analytical" / "sales.csv",
    low_memory=False,
    parse_dates=["Date"]
)

print("Loaded tables:")
print({
    "orders": orders.shape,
    "order_items": order_items.shape,
    "payments": payments.shape,
    "returns": returns.shape,
    "products": products.shape,
    "customers": customers.shape,
    "geography": geography.shape,
    "web_traffic": web_traffic.shape,
    "sales": sales.shape,
})

Loaded tables:
{'orders': (646945, 8), 'order_items': (714669, 7), 'payments': (646945, 4), 'returns': (39939, 7), 'products': (2412, 8), 'customers': (121930, 7), 'geography': (39948, 4), 'web_traffic': (3652, 7), 'sales': (3833, 3)}


In [19]:
# Q1: Median inter-order gap (ngay) cho khach hang co hon 1 don
orders_q1 = orders.sort_values(["customer_id", "order_date"]).copy()

# Tinh khoang cach ngay giua 2 lan mua lien tiep trong tung customer
orders_q1["inter_order_gap_days"] = orders_q1.groupby("customer_id")["order_date"].diff().dt.days

# Chi giu cac gap hop le (tu customer co it nhat 2 don)
gaps = orders_q1["inter_order_gap_days"].dropna()
q1_median_gap = gaps.median()

print("Q1 - Median inter-order gap (days):", q1_median_gap)

#Q1 : chọn C (144 xap xi len 180)

Q1 - Median inter-order gap (days): 144.0


In [ ]:
# Q2: Segment co mean gross margin ratio cao nhat
products_q2 = products.copy()

# Gross margin ratio = (price - cogs) / price
products_q2["gross_margin_ratio"] = (products_q2["price"] - products_q2["cogs"]) / products_q2["price"]

q2_by_segment = (
    products_q2.groupby("segment", dropna=False)["gross_margin_ratio"]
    .mean()
    .sort_values(ascending=False)
)

print("Q2 - Segment co gross margin ratio cao nhat:")
print(q2_by_segment.head(1))

print("\nTop segments:")
print(q2_by_segment.head())

#Q2 chon D

Q2 - Segment co gross margin ratio cao nhat:
segment
Standard    0.313442
Name: gross_margin_ratio, dtype: float64

Top segments:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Name: gross_margin_ratio, dtype: float64


In [ ]:
# Q3: Ly do tra hang pho bien nhat cho category = Streetwear
returns_q3 = returns.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
 )

streetwear_returns = returns_q3[returns_q3["category"] == "Streetwear"]
q3_reason_counts = streetwear_returns["return_reason"].value_counts(dropna=False)

print("Q3 - Return reason xuat hien nhieu nhat (Streetwear):")
print(q3_reason_counts.head(1))
print("\nTat ca ly do:")
print(q3_reason_counts)

#Q3 chon B

Q3 - Return reason xuat hien nhieu nhat (Streetwear):
return_reason
wrong_size    7626
Name: count, dtype: int64

Tat ca ly do:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


In [ ]:
# Q4: Traffic source co mean bounce_rate thap nhat
q4_by_source = (
    web_traffic.groupby("traffic_source", dropna=False)["bounce_rate"]
    .mean()
    .sort_values(ascending=True)
)

print("Q4 - Traffic source co mean bounce_rate thap nhat:")
print(q4_by_source.head(1))
print("\nTat ca traffic_source:")
print(q4_by_source)

#Q4 chon C

Q4 - Traffic source co mean bounce_rate thap nhat:
traffic_source
email_campaign    0.004458
Name: bounce_rate, dtype: float64

Tat ca traffic_source:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


In [42]:
# Q5: Ty le % dong order_items co promo_id khong null
q5_total_rows = len(order_items)

# So dong co promo_id + so dong co promo_id_2 - so dong co ca hai
q5_promo_rows = (
    order_items["promo_id"].notna().sum()
    + order_items["promo_id_2"].notna().sum()
    - (order_items["promo_id"].notna() & order_items["promo_id_2"].notna()).sum()
)

print(f"Q5 - So dong co promo_id: {q5_promo_rows}")

q5_ratio_percent = (q5_promo_rows / q5_total_rows) * 100 if q5_total_rows > 0 else np.nan

print(f"Q5 - Ty le dong co promo_id: {q5_ratio_percent:.2f}%")
print({"promo_rows": int(q5_promo_rows), "total_rows": int(q5_total_rows)})

#Q5 chon C

Q5 - So dong co promo_id: 276316
Q5 - Ty le dong co promo_id: 38.66%
{'promo_rows': 276316, 'total_rows': 714669}


In [ ]:
# Q6: Age group co so don hang trung binh / customer cao nhat
customers_q6 = customers[["customer_id", "age_group"]].drop_duplicates()
orders_q6 = orders[["order_id", "customer_id"]].copy()

q6_join = customers_q6.merge(orders_q6, on="customer_id", how="left")
q6_stats = q6_join.groupby("age_group", dropna= False).agg(
    total_orders=("order_id", "count"),
    total_customers=("customer_id", "nunique"),
)
q6_stats["avg_orders_per_customer"] = q6_stats["total_orders"] / q6_stats["total_customers"]
q6_stats = q6_stats.sort_values("avg_orders_per_customer", ascending=False)

print("Q6 - Age group cao nhat theo avg_orders_per_customer:")
print(q6_stats.head(1))
print("\nTat ca age_group:")
print(q6_stats)

#Q6 chon A

Q6 - Age group cao nhat theo avg_orders_per_customer:
           total_orders  total_customers  avg_orders_per_customer
age_group                                                        
55+               72760            13457                 5.406851

Tat ca age_group:
           total_orders  total_customers  avg_orders_per_customer
age_group                                                        
55+               72760            13457                 5.406851
45-54            124138            23172                 5.357241
35-44            170368            31920                 5.337343
25-34            190622            36342                 5.245226
18-24             89057            17039                 5.226656


In [ ]:
# Q7: Region co tong doanh thu cao nhat
# Luu y: sales.csv dang o cap do ngay (Date, Revenue), khong co region/order_id.
# Cach lam ben duoi: uoc luong doanh thu theo region bang cach phan bo Revenue moi ngay theo ti trong so don cua region trong ngay do.

orders_q7 = orders[["order_id", "order_date", "zip"]].copy()
orders_q7["order_day"] = orders_q7["order_date"].dt.date

orders_geo_q7 = orders_q7.merge(geography[["zip", "region"]], on="zip", how="left")

# Dem so don theo ngay-region
daily_region_orders = (
    orders_geo_q7.groupby(["order_day", "region"]).size().rename("region_orders").reset_index()
)

# Tong so don theo ngay
daily_total_orders = (
    orders_geo_q7.groupby("order_day").size().rename("total_orders").reset_index()
)

daily_share = daily_region_orders.merge(daily_total_orders, on="order_day", how="left")
daily_share["order_share"] = daily_share["region_orders"] / daily_share["total_orders"]

sales_q7 = sales[["Date", "Revenue"]].copy()
sales_q7["order_day"] = sales_q7["Date"].dt.date

# Gan doanh thu ngay cho region theo ti trong don
region_revenue_q7 = daily_share.merge(sales_q7[["order_day", "Revenue"]], on="order_day", how="inner")
region_revenue_q7["estimated_region_revenue"] = region_revenue_q7["order_share"] * region_revenue_q7["Revenue"]

q7_result = (
    region_revenue_q7.groupby("region")["estimated_region_revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Q7 - Region co tong doanh thu uoc luong cao nhat:")
print(q7_result.head(1))
print("\nTat ca region:")
print(q7_result)

#Q7 chon C


#noticed

Q7 - Region co tong doanh thu uoc luong cao nhat:
region
East    7.480137e+09
Name: estimated_region_revenue, dtype: float64

Tat ca region:
region
East       7.480137e+09
Central    4.730484e+09
West       4.219855e+09
Name: estimated_region_revenue, dtype: float64


In [ ]:
# Q8: Trong cac don cancelled, payment_method nao pho bien nhat
cancelled_orders = orders[orders["order_status"] == "cancelled"].copy()
q8_payment_counts = cancelled_orders["payment_method"].value_counts(dropna=False)

print("Q8 - Payment method duoc dung nhieu nhat trong don cancelled:")
print(q8_payment_counts.head(1))
print("\nTat ca payment methods:")
print(q8_payment_counts)

#Q8 chon A

Q8 - Payment method duoc dung nhieu nhat trong don cancelled:
payment_method
credit_card    28452
Name: count, dtype: int64

Tat ca payment methods:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


In [ ]:
# Q9: Size nao (S, M, L, XL) co return rate cao nhat
# Dinh nghia: so ban ghi returns / so dong order_items theo tung size

sizes = ["S", "M", "L", "XL"]

returns_q9 = returns.merge(products[["product_id", "size"]], on="product_id", how="left")
order_items_q9 = order_items.merge(products[["product_id", "size"]], on="product_id", how="left")

returns_count_by_size = returns_q9["size"].value_counts().reindex(sizes, fill_value=0)
order_items_count_by_size = order_items_q9["size"].value_counts().reindex(sizes, fill_value=0)

q9_return_rate = returns_count_by_size / order_items_count_by_size.replace(0, np.nan)
q9_return_rate = q9_return_rate.sort_values(ascending=False)

print("Q9 - Size co return rate cao nhat:")
print(q9_return_rate.head(1))
print("\nReturn rate theo size:")
print(q9_return_rate)

#Q9 chon A

Q9 - Size co return rate cao nhat:
size
S    0.056515
Name: count, dtype: float64

Return rate theo size:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64


In [ ]:
# Q10: Installments nao co mean payment_value cao nhat
q10_by_installments = (
    payments.groupby("installments", dropna=False)["payment_value"]
    .mean()
    .sort_values(ascending=False)
)

print("Q10 - Installments co mean payment_value cao nhat:")
print(q10_by_installments.head(1))
print("\nTat ca installments:")
print(q10_by_installments)

#Q10 chon C

Q10 - Installments co mean payment_value cao nhat:
installments
6    24446.654403
Name: payment_value, dtype: float64

Tat ca installments:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


In [12]:
# Tong hop dap an nhanh sau khi da chay Q1-Q10
answers = {
    "Q1_median_inter_order_gap_days": float(q1_median_gap),
    "Q2_best_segment": q2_by_segment.idxmax(),
    "Q3_top_streetwear_return_reason": q3_reason_counts.idxmax(),
    "Q4_lowest_bounce_source": q4_by_source.idxmin(),
    "Q5_promo_usage_percent": float(q5_ratio_percent),
    "Q6_best_age_group": q6_stats["avg_orders_per_customer"].idxmax(),
    "Q7_top_region_estimated_revenue": q7_result.idxmax(),
    "Q8_top_cancelled_payment_method": q8_payment_counts.idxmax(),
    "Q9_highest_return_rate_size": q9_return_rate.idxmax(),
    "Q10_best_installments": int(q10_by_installments.idxmax()),
}

print(pd.Series(answers))

Q1_median_inter_order_gap_days              144.0
Q2_best_segment                          Standard
Q3_top_streetwear_return_reason        wrong_size
Q4_lowest_bounce_source            email_campaign
Q5_promo_usage_percent                  38.663493
Q6_best_age_group                             55+
Q7_top_region_estimated_revenue              East
Q8_top_cancelled_payment_method       credit_card
Q9_highest_return_rate_size                     S
Q10_best_installments                           6
dtype: object
